# EDA — Base de Dados de Eficiência Energética (London Smart Meters)

**Objetivo deste notebook:**
- Importar os dados brutos do dataset Low Carbon London (smart meters)
- Explorar e limpar as séries temporais de consumo
- Derivar as variáveis obrigatórias do projeto a partir do dado real
- Combinar com variáveis sintéticas correlacionadas (equipamentos, tipo de imóvel)
- Gerar o dataset final: `energiai_dataset_v1.csv`

**Fonte dos dados:** Low Carbon London Project (UK Power Networks, 2011–2014),
distribuído via London Datastore, licença aberta (CC-BY). Leituras de consumo em
blocos de 30 minutos, por domicílio (`LCLid`).

**Variáveis obrigatórias do projeto e origem de cada uma:**

| Variável | Origem |
|---|---|
| `consumo_kwh` | Real — derivado da série de 30 min |
| `horas_alto_consumo` | Real — derivado da série de 30 min |
| `uso_horario_pico` | Real — derivado da série de 30 min |
| `quantidade_equipamentos` | Sintética, correlacionada ao consumo real |
| `tipo_imovel` | Sintética, correlacionada ao consumo real |


## Etapa 1 — Importando as bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None) #evitar exibição truncada

## Etapa 2 — Extração dos arquivos brutos (referência)

Os dados brutos (`LCL-June2015v2_*.csv`) vêm de um zip com 168 arquivos, cada um
contendo leituras de ~30 domicílios em blocos de 30 minutos. Para este projeto,
extraímos 17 arquivos (índices `0` a `16`), totalizando **489 domicílios únicos**
— volume suficiente para treinar os modelos supervisionados.
Os dados podem ser baixados acessando o link https://data.london.gov.uk/dataset/smartmeter-energy-consumption-data-in-london-households-vqm0d e baixando o arquivo zip contendo os 168 arquivos.

> Célula de referência/documentação — não precisa ser executada de novo se os CSVs
> já estão em `data/raw/` (essa pasta está no `.gitignore`, então esta célula é o
> registro de como qualquer pessoa pode reconstruí-la a partir do zip
> original). Basta executar o código documentado abaixo:


```
import zipfile
import os

caminho_zip = "~/Downloads/low-carbon-london-data-168-files.zip"  # ajuste para o seu caminho
caminho_zip = os.path.expanduser(caminho_zip)
pasta_destino = "../data/raw/"

indices_desejados = range(0, 17)  # arquivos _0 até _16 (17 arquivos)

with zipfile.ZipFile(caminho_zip) as z:
    nomes = z.namelist()
    extraidos = 0
    for i in indices_desejados:
        alvo = [n for n in nomes if n.endswith(f"_{i}.csv")]
        if alvo:
            z.extract(alvo[0], pasta_destino)
            extraidos += 1
        else:
            print(f"Aviso: índice {i} não encontrado no zip")
    print(f"{extraidos} arquivos extraídos para {pasta_destino}")
```


## Etapa 3 — Carregamento dos dados brutos

Carregamos todos os arquivos extraídos em `data/raw/` com `glob` (evita listar os
17 nomes manualmente), concatenando em um único DataFrame. Cada linha representa
uma leitura de consumo (kWh) de um domicílio em um bloco de 30 minutos.


In [ ]:
arquivos = sorted(glob.glob("../data/raw/LCL-June2015v2_*.csv"))
print(f"Arquivos encontrados: {len(arquivos)}")
for a in arquivos:
    print(" -", a)

bruto = pd.concat([pd.read_csv(f) for f in arquivos], ignore_index=True)

print("\nShape:", bruto.shape)
print("\nColunas:", bruto.columns.tolist())
print("\nTipos de dado:")
print(bruto.dtypes)
bruto.head()


## Etapa 4 — Limpeza inicial e diagnóstico

Verificação de:
- Nomes de colunas
- Quantidade de domicílios únicos (meta: ~400–500)
- Valores nulos (aparecem como string `"Null"`, não como `NaN`)
- Período coberto pelos dados
- Tipos de tarifa presentes (`stdorToU`)

**Resultado obtido:** 489 domicílios únicos, 489 valores `"Null"` (1 por domicílio
— provável artefato de calibração inicial do medidor), tarifa 100% `Std`, cobrindo
de nov/2011 a fev/2014.


In [ ]:
# remove espaços em branco extras dos nomes de coluna
bruto.columns = bruto.columns.str.strip()
print("Colunas corrigidas:", bruto.columns.tolist())

# quantidade de domicílios únicos
n_domicilios = bruto["LCLid"].nunique()
print("\nDomicílios únicos (LCLid):", n_domicilios)

# quantidade de valores 'Null' (string) existentes na coluna de consumo
print("Valores 'Null' na coluna de consumo:", (bruto["KWH/hh (per half hour)"] == "Null").sum())

# período coberto
print("\nPrimeira data:", bruto["DateTime"].min())
print("Última data:", bruto["DateTime"].max())

# tipos de tarifa
print("\nValores de stdorToU:")
print(bruto["stdorToU"].value_counts())


## Etapa 5 — Remoção de nulos e conversão de tipos

Removemos os registros com `"Null"` na coluna de consumo — justificativa: menos de
0,003% dos dados, padrão estrutural (1 por domicílio, provável artefato de
calibração inicial do medidor), sem impacto estatístico relevante em nenhuma
agregação por domicílio. Optamos por remover em vez de preencher com 0, pois
`0 kWh` afirmaria um consumo real que não temos evidência de ter ocorrido.

Em seguida, convertemos `KWH/hh` para `float` e `DateTime` para `datetime`
(necessário para extrair hora do dia e mês na próxima etapa). Por fim, checamos se
existe consumo negativo (não deveria existir).


In [ ]:
print("Linhas antes:", len(bruto))
bruto = bruto[bruto["KWH/hh (per half hour)"] != "Null"].copy()
print("Linhas depois:", len(bruto))

bruto["KWH/hh (per half hour)"] = bruto["KWH/hh (per half hour)"].astype(float)
bruto["DateTime"] = pd.to_datetime(bruto["DateTime"])

print("\nTipos de dado após conversão:")
print(bruto.dtypes)

print("\nEstatísticas do consumo (kWh por bloco de 30min):")
print(bruto["KWH/hh (per half hour)"].describe())

negativos = (bruto["KWH/hh (per half hour)"] < 0).sum()
print(f"\nValores negativos de consumo: {negativos}")


## Etapa 6 — Derivação das variáveis reais por domicílio

Transformamos a série de 30 em 30 minutos (1 linha por leitura) em 1 linha por
domicílio, calculando as 3 variáveis que vêm de dado real:

- **consumo_kwh**: mediana do consumo mensal por domicílio (robusta a meses atípicos)
- **horas_alto_consumo**: média diária de blocos de 30 min acima da própria média
  daquele domicílio (parâmetro individual, não uma média geral). *Nota: por ser
  relativa a cada casa, essa variável mede a variabilidade do próprio consumo, não
  a escala — por isso tem baixa correlação com consumo_kwh (ver Etapa 8), o que é
  esperado e desejável (informação não-redundante para o modelo)*
- **proporcao_pico**: proporção do consumo entre 18h-21h (usada para derivar
  uso_horario_pico na próxima etapa)


In [ ]:
bruto["hora"] = bruto["DateTime"].dt.hour
bruto["ano_mes"] = bruto["DateTime"].dt.to_period("M")

# consumo_kwh: mediana do consumo mensal por domicílio
consumo_mensal = bruto.groupby(["LCLid", "ano_mes"])["KWH/hh (per half hour)"].sum()
consumo_kwh = consumo_mensal.groupby("LCLid").median().rename("consumo_kwh")

# horas_alto_consumo: blocos acima da própria média do domicílio, em horas/dia
media_por_domicilio = bruto.groupby("LCLid")["KWH/hh (per half hour)"].transform("mean")
bruto["acima_media"] = bruto["KWH/hh (per half hour)"] > media_por_domicilio

blocos_acima = bruto.groupby("LCLid")["acima_media"].sum()
dias_observados = bruto.groupby("LCLid")["DateTime"].apply(lambda s: s.dt.date.nunique())
horas_alto_consumo = (blocos_acima / dias_observados / 2).rename("horas_alto_consumo")

# proporção do consumo no horário de pico (18h-21h)
total_por_domicilio = bruto.groupby("LCLid")["KWH/hh (per half hour)"].sum()
consumo_pico = bruto[bruto["hora"].between(18, 20)].groupby("LCLid")["KWH/hh (per half hour)"].sum()
proporcao_pico = (consumo_pico / total_por_domicilio).fillna(0)

print("Domicílios processados:", consumo_kwh.shape[0])


## Etapa 7 — Ajuste dos limiares por quantil (pico e outliers)

Duas decisões com o mesmo princípio: usar a distribuição real observada
para definir o corte, em vez de um número fixo escolhido a priori.

**uso_horario_pico**: um limiar fixo testado inicialmente (15%) resultou em 85%
dos domicílios classificados como `True` — pouco poder discriminativo. A
distribuição real da proporção de consumo no pico segue formato próximo do
normal, concentrada entre 15%-24%, sem separação natural entre grupos (não existe
um corte "óbvio" nos dados). Adotamos o **quantil 70** como corte: os ~30% de
domicílios com maior concentração no horário de pico são marcados como `True`.

**consumo_kwh — outliers**: em vez de um limite fixo (ex: 20-1200 kWh, testado
antes), removemos os domicílios fora da faixa entre os **percentis 1 e 99** — mais
robusto à assimetria (cauda longa) da distribuição de consumo do que um valor fixo
definido de antemão.

*Por que quantil/percentil em vez de número fixo, no geral: um valor fixo assume
que já sabemos de antemão o que é "normal"; se a suposição estiver errada (como
aconteceu com os 15%), a variável resultante fica com pouca ou nenhuma capacidade
de separar os domicílios. O corte por posição relativa na distribuição sempre
garante uma divisão informativa, qualquer que seja a distribuição real dos dados.*


In [ ]:
# --- ajuste 1: uso_horario_pico via quantil 70 ---
corte_pico = proporcao_pico.quantile(0.70)
print(f"Corte usado (quantil 70): {corte_pico:.4f}")

uso_horario_pico = (proporcao_pico > corte_pico).rename("uso_horario_pico")
print("Proporção de True:", round(uso_horario_pico.mean(), 3))

# --- ajuste 2: outliers de consumo_kwh via percentil 1-99 ---
p1 = consumo_kwh.quantile(0.01)
p99 = consumo_kwh.quantile(0.99)
print(f"\nFaixa aceita de consumo_kwh: {p1:.1f} a {p99:.1f}")

# --- reconstrói a tabela final com os ajustes ---
dados_reais = pd.concat([consumo_kwh, horas_alto_consumo, uso_horario_pico], axis=1).reset_index()
dados_reais = dados_reais.rename(columns={"LCLid": "meter_id"})

antes = len(dados_reais)
dados_reais = dados_reais[dados_reais["consumo_kwh"].between(p1, p99)].reset_index(drop=True)
print(f"\nDomicílios antes do corte de outliers: {antes}")
print(f"Domicílios depois do corte de outliers: {len(dados_reais)}")

print("\nProporção final uso_horario_pico=True:", round(dados_reais["uso_horario_pico"].mean(), 3))
dados_reais.describe()


## Etapa 8 — Variáveis sintéticas correlacionadas: quantidade_equipamentos e tipo_imovel

A base pública utilizada não contém inventário de
eletrodomésticos ou tipo de imóvel por domicílio. Essas duas variáveis são geradas
por fórmula, **correlacionadas ao consumo_kwh real** (não sorteadas de forma
independente) — uma variável independente do consumo não teria relação causal
nenhuma com o restante da base, o que não reflete a realidade (mais equipamentos
tende a significar mais consumo).

- **quantidade_equipamentos**: cresce proporcionalmente ao consumo real
  normalizado, mais um termo de ruído aleatório
- **tipo_imovel**: inferido pela faixa de consumo (Apartamento / Casa / Comercio),
  com ~8% de ruído para evitar um corte perfeitamente determinístico

**Decisão técnica:** optamos por não vincular `tipo_imovel` e
`quantidade_equipamentos` diretamente entre si (ex: regra "+X equipamentos se for
Comercio"), pois ambas já derivam do mesmo `consumo_kwh` — vinculá-las diretamente
geraria redundância artificial (multicolinearidade).

Também descartamos a classificação **ACORN** (disponível na base de
Londres) como proxy de `tipo_imovel` — porque ACORN é
classificação socioeconômica de bairro (perfil de consumo/estilo de vida), não
tipo de construção; usá-la misturaria classe social com tipo de imóvel, o que é
estatisticamente inválido.


In [ ]:
rng = np.random.default_rng(42)  # seed fixa = reproduzível
n = len(dados_reais)

consumo_norm = dados_reais["consumo_kwh"] / dados_reais["consumo_kwh"].max()

# quantidade_equipamentos: baseline cresce com o consumo real
baseline_equip = 2 + consumo_norm * 25

# tipo_imovel: inferido pela faixa de consumo (não é dado real - documentado acima)
limite_casa = dados_reais["consumo_kwh"].quantile(0.55)
limite_apto = dados_reais["consumo_kwh"].quantile(0.85)

tipo = np.where(
    dados_reais["consumo_kwh"] <= limite_apto,
    np.where(dados_reais["consumo_kwh"] <= limite_casa, "Apartamento", "Casa"),
    "Comercio",
)
mask_ruido = rng.random(n) < 0.08
tipo[mask_ruido] = rng.choice(["Casa", "Apartamento", "Comercio"], size=mask_ruido.sum())
dados_reais["tipo_imovel"] = tipo

print("Distribuição de tipo_imovel:")
print(dados_reais["tipo_imovel"].value_counts())


### 8.1 Diagnóstico: ruído fixo gerou um viés sistemático

Uma primeira versão usou ruído com desvio-padrão **fixo** (2,5) para todos os
domicílios. Ao validar a qualidade da variável gerada, encontramos dois sinais de
alerta:

1. **Correlação `horas_alto_consumo` com o resto é baixa (~0,10)** — esperado, não
   é problema (ela mede variabilidade relativa da própria casa, não escala; baixa
   correlação com as demais significa que carrega informação não-redundante)
2. **11 de 479 domicílios (2,3%) com "consumo por equipamento" > 100 kWh** — nesses
   casos, `quantidade_equipamentos` batia no piso de 1, quase todos em Apartamento
   (a categoria com a baseline mais baixa). Isso não era ruído aleatório bem
   distribuído: era um viés sistemático — o mesmo desvio-padrão fixo tinha muito
   mais chance de "estourar para baixo" em domicílios com baseline já baixa.


In [ ]:
quantidade_equipamentos_v1 = np.clip(
    baseline_equip + rng.normal(0, 2.5, n), 1, 30
).round().astype(int)

consumo_por_equip_v1 = dados_reais["consumo_kwh"] / quantidade_equipamentos_v1
extremos_v1 = (consumo_por_equip_v1 > 100).sum()
print(f"Casos extremos com ruído FIXO: {extremos_v1} de {n} ({extremos_v1/n*100:.1f}%)")


### 8.2 Correção: ruído proporcional à baseline

Trocamos o ruído de desvio-padrão fixo por um ruído **proporcional** à baseline
esperada de cada domicílio (25% da própria baseline) — isso reduz a chance de
"estourar" o piso especificamente em quem já começa com baseline baixa, sem
remover a aleatoriedade necessária para a variável não ficar 100% determinística.


In [ ]:
ruido_proporcional = rng.normal(0, baseline_equip * 0.25)
quantidade_equipamentos = baseline_equip + ruido_proporcional
dados_reais["quantidade_equipamentos"] = np.clip(quantidade_equipamentos, 1, 30).round().astype(int)

consumo_por_equip = dados_reais["consumo_kwh"] / dados_reais["quantidade_equipamentos"]
extremos = dados_reais[consumo_por_equip > 100]
print(f"Casos extremos após correção: {len(extremos)} de {len(dados_reais)} ({len(extremos)/len(dados_reais)*100:.1f}%)")

print("\nquantidade_equipamentos por tipo_imovel (validação - a ordem esperada deve se manter):")
print(dados_reais.groupby("tipo_imovel")["quantidade_equipamentos"].agg(["mean", "median", "std"]))


## Etapa 9 — Rotulagem da categoria de eficiência e custo estimado

Calculamos um score ponderado sobre as variáveis (consumo_kwh normalizado: peso
0,5; consumo por equipamento: peso 0,2; horas_alto_consumo normalizado: peso 0,2;
uso_horario_pico: peso 0,1), ordenamos os domicílios por esse score e cortamos por
**quantil** — os 30% com menor score viram "Eficiente", os 40% do meio
"Moderado", os 30% com maior score "Ineficiente".

**Por que quantil em vez de limiares fixos:** limiares fixos testados
anteriormente geraram classes desbalanceadas (ex: 51% Moderado, 14% Ineficiente),
prejudicial para o treinamento do modelo. O corte por quantil garante 3 classes
com volume comparável, o que facilita a avaliação por métricas como F1-score por
classe. O enunciado permite que a equipe defina os critérios de categorização,
desde que justificados — um corte por quantil documentado é uma justificativa
objetiva e reproduzível.

Por fim, aplicamos a tarifa de referência do projeto (R$ 0,75/kWh) para gerar
`custo_estimado_mensal`. *Nota para a etapa de modelagem: `custo_estimado_mensal`
tem correlação perfeita (1,0) com `consumo_kwh`, por ser uma transformação linear
direta — não deve ser usada como feature de treino, é calculada à parte, como
saída da API, não entrada.*


In [ ]:
TARIFA_KWH = 0.75  # valor de referência do projeto

consumo_por_equip = dados_reais["consumo_kwh"] / dados_reais["quantidade_equipamentos"].clip(lower=1)

score = (
    0.5 * (dados_reais["consumo_kwh"] / dados_reais["consumo_kwh"].max())
    + 0.2 * (consumo_por_equip / consumo_por_equip.max())
    + 0.2 * (dados_reais["horas_alto_consumo"] / dados_reais["horas_alto_consumo"].max())
    + 0.1 * dados_reais["uso_horario_pico"].astype(int)
)

dados_reais["categoria"] = pd.qcut(
    score, q=[0, 0.30, 0.70, 1.0], labels=["Eficiente", "Moderado", "Ineficiente"]
).astype(str)

dados_reais["custo_estimado_mensal"] = (dados_reais["consumo_kwh"] * TARIFA_KWH).round(2)

print("Distribuição de categoria:")
print(dados_reais["categoria"].value_counts())
print("\nDistribuição cruzada categoria x tipo_imovel:")
print(pd.crosstab(dados_reais["categoria"], dados_reais["tipo_imovel"]))

dados_reais.head(10)


## Etapa 10 — Exportação do dataset final

Salvamos o resultado em `data/processed/energiai_dataset_v1.csv` — o arquivo que
alimenta o treinamento dos modelos supervisionados.

**Nota para quem for ler este CSV depois:** o formato CSV não preserva tipos de
dado — ao reabrir com `pd.read_csv`, a coluna `uso_horario_pico` volta como texto
(`"True"`/`"False"`), não como booleano. É necessário reconverter com
`.astype(bool)` após a leitura.


In [ ]:
caminho_saida = "../data/processed/energiai_dataset_v1.csv"
dados_reais.to_csv(caminho_saida, index=False)

print(f"Dataset salvo em: {caminho_saida}")
print(f"Shape final: {dados_reais.shape}")
print("\nTipos de dado:")
print(dados_reais.dtypes)


## Etapa 11 — Análise Visual dos Dados 



### 11.1 Consumo mensal de energia

Essa análise mostra a forma de distribuição do consumo real entre os 479 domicílios, evidenciando como os valores se espalham. 

Observa-se uma **distribuição assimétrica à direita**, com maior concentração dos domicílios consumindo entre 150 e 250 kWh, enquanto uma quantidade menor apresenta consumos significativamente mais elevados, formando uma cauda que se estende para valores superiores a 1.100 kWh. **Esse comportamento confirma a presença de valores extremos e reforça a adequação do uso de percentis para o tratamento de outliers**, em vez de métodos baseados na suposição de uma distribuição aproximadamente normal.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(dados_reais["consumo_kwh"], bins=30, kde=True, color="red")
plt.title("Distribuição do consumo mensal (kWh) por domicílio")
plt.xlabel("Consumo mensal (kWh)")
plt.ylabel("Número de domicílios")
plt.show()

### 11.2 Distribuição do consumo por categoria

A análise foi realizada para verificar se as categorias apresentam uma boa separação em relação ao consumo, variável que possui o maior peso no cálculo do score (0,5).

Observa-se uma clara diferença entre os grupos, com medianas de aproximadamente 140 kWh para Eficiente, 240 kWh para Moderado e 450 kWh para Ineficiente. Há pouca sobreposição entre as distribuições de Eficiente e Ineficiente, **indicando que as categorias discriminam bem os domicílios de acordo com seu nível de consumo**, conforme esperado pelo critério utilizado na classificação.

In [ ]:
plt.figure(figsize=(8, 5))
ordem = ["Eficiente", "Moderado", "Ineficiente"]
sns.boxplot(data=dados_reais, x="categoria", y="consumo_kwh", order=ordem, hue="categoria", palette="Blues", legend=False)
plt.title("Consumo mensal (kWh) por categoria de eficiência")
plt.xlabel("Categoria")
plt.ylabel("Consumo mensal (kWh)")
plt.show()

### 11.3 Distribuição dos domicílios por categoria de eficiência

A análise foi realizada para verificar o **balanceamento entre as classes**, uma vez que uma diferença muito grande na quantidade de amostras poderia introduzir viés nas análises ou previsões do modelo.

Observa-se uma **distribuição relativamente equilibrada** entre as categorias, com 144 domicílios Eficientes, 191 Moderados e 144 Ineficientes. Esses valores correspondem aproximadamente a 30%, 40% e 30% da amostra, respectivamente, confirmando o balanceamento esperado obtido por meio do corte por quantis. Dessa forma, não há uma classe significativamente maior que as demais.


In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=dados_reais, x="categoria", order=ordem, hue="categoria", palette="Blues", legend=False)
plt.title("Quantidade de domicílios por categoria de eficiência")
plt.xlabel("Categoria")
plt.ylabel("Número de domicílios")
plt.show()

### 11.4 Correlação entre as variáveis

Essa análise é importante para a **etapa de modelagem**, pois ajuda a identificar variáveis redundantes que podem fornecer essencialmente a mesma informação ao modelo.

Destacam-se dois pontos principais. A variável `custo_estimado_mensal` apresenta correlação perfeita (1,00) com `consumo_kwh`, o que é esperado, pois seu valor é obtido diretamente a partir do consumo multiplicado pela tarifa fixa de R$ 0,75. Portanto, `custo_estimado_mensal` não deve ser utilizada como feature de treinamento, sendo calculada separadamente como saída da API. Já `horas_alto_consumo` apresenta correlação fraca (≈ 0,10) com as demais variáveis, indicando que possui informações pouco redundantes e potencialmente complementares para a modelagem.

In [ ]:
plt.figure(figsize=(8, 6))
colunas_numericas = ["consumo_kwh", "horas_alto_consumo", "quantidade_equipamentos", "custo_estimado_mensal"]
matriz_corr = dados_reais[colunas_numericas].corr()

sns.heatmap(matriz_corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlação entre variáveis numéricas")
plt.show()

### 11.5 Poder de discriminação da variável `horas_alto_consumo`

A análise busca verificar se a variável `horas_alto_consumo`, que possui peso 0,20 no cálculo do score, apresenta capacidade real de diferenciar os grupos.

As medianas dos três grupos são bastante próximas, aproximadamente 7,5 horas para Eficiente, 8,0 para Moderado e 8,2 para Ineficiente, além de haver uma considerável sobreposição entre as distribuições. Isso indica que `horas_alto_consumo` possui **baixo poder de discriminação entre as categorias**, apresentando uma separação significativamente menor do que a observada para `consumo_kwh`. Portanto, apesar de possuir um peso relevante na fórmula do score, essa variável contribui de forma mais limitada para diferenciar os níveis de eficiência dos domicílios.



In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=dados_reais, x="categoria", y="horas_alto_consumo", order=ordem, hue="categoria", palette="Blues", legend=False)
plt.title("Horas de alto consumo (por dia) por categoria de eficiência")
plt.xlabel("Categoria")
plt.ylabel("Horas de alto consumo (média diária)")
plt.show()

### 11.6 Uso de energia no horário de pico

A análise busca verificar se o uso concentrado de energia no horário de pico está associado a níveis maiores de ineficiência e se o peso de 0,10 atribuído à variável no score é coerente com seu comportamento nos dados.

Observa-se **uma tendência crescente e bem definida entre as categorias**: apenas 4% dos domicílios Eficientes apresentam uso concentrado no horário de pico, enquanto essa proporção aumenta para 34% nos Moderados e 51% nos Ineficientes. **Esse comportamento demonstra que a variável possui bom poder de discriminação entre as categorias, indicando uma associação clara entre o uso no horário de pico e a maior ineficiência**. Os resultados, portanto, justificam a inclusão da variável no score, mesmo com seu menor peso (0,10).

In [ ]:
proporcoes = dados_reais.groupby("categoria")["uso_horario_pico"].mean().reindex(ordem)

plt.figure(figsize=(7, 5))
proporcoes.plot(kind="bar", color=["#8fc7fc", "#6baed6", "#2171b5"])
plt.title("Proporção de domicílios com uso concentrado no horário de pico, por categoria")
plt.xlabel("Categoria")
plt.ylabel("Proporção com uso_horario_pico = True")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.show()

### 11.7 Relação entre quantidade de equipamentos e tipo de imóvel

A análise foi realizada para verificar se a relação esperada entre essas variáveis surge naturalmente a partir dos dados, sem a necessidade de criar uma regra explícita vinculando `tipo_imovel` a `quantidade_equipamentos`. Essa abordagem evita introduzir redundância, uma vez que ambas as variáveis possuem relação indireta com o consumo real de energia.

Observa-se claramente a ordem esperada de quantidade de equipamentos entre os tipos de imóvel: Apartamento (mediana ≈ 6 equipamentos), Casa (≈ 9–10) e Comércio (≈ 15–20). Há pouca sobreposição entre os extremos, indicando uma diferença consistente entre os grupos. **O resultado demonstra que a relação esperada emerge naturalmente dos dados, por meio da relação indireta com o consumo**, validando a decisão de não estabelecer uma regra direta entre `tipo_imovel` e `quantidade_equipamentos`.

In [ ]:
plt.figure(figsize=(8, 5))
ordem_imovel = ["Apartamento", "Casa", "Comercio"]
sns.boxplot(data=dados_reais, x="tipo_imovel", y="quantidade_equipamentos", order=ordem_imovel, hue="tipo_imovel", palette="Reds", legend=False)
plt.title("Quantidade de equipamentos por tipo de imóvel")
plt.xlabel("Tipo de imóvel")
plt.ylabel("Quantidade de equipamentos")
plt.show()